> ⚠️ **机器翻译提示**：本 notebook 由机器翻译生成，未经人工校对，如发现翻译错误敬请谅解（代码与计算结果和越南语原版完全一致，只有文字说明是机器翻译）。

# 第08单元 — Pearson相关与偏相关
第08单元专属 notebook —— 重现Pearson r + Spearman rho（班级数据）以及偏相关（真实PISA数据）—— 偏相关部分 PSPP 2.1.1 不支持（`PARTIAL CORR is not yet implemented`），只能在此运行。

**本 notebook 力求自成一体，无需另外打开网页** —— 每种方法在运行代码前都有"这个方法用来回答什么问题"的说明，运行后都有"结果有什么含义"的解读。

## 0. Pearson相关系数用来回答什么问题？
Pearson相关系数（r）回答的问题是：**"这两个变量是否倾向于沿一条直线同增同减，程度有多强？"** 仅此而已。具体来说：

- **符号（+/−）**：同向变化（正）还是反向变化（负）。
- **大小 |r|**（0到1）：一条直线拟合数据点云的程度如何 —— 不是"变量A对变量B的影响有多大"。
- **r 不说明任何因果关系** —— 只描述一种共同出现的线性模式。r很高可能是A导致B、B导致A、第三个变量C同时导致两者，也可能纯属偶然（样本量小）。
- **对本 notebook 后面部分最重要的性质：r 是对称的** —— r(学习时长, 数学成绩) = r(数学成绩, 学习时长)，无论按哪个方向计算结果都一样。但一旦开始讨论"X增加多少，Y就增加多少"，这一点很容易造成混淆 —— 详见第5节。

In [ ]:
# 先运行这一格（Colab：点击 ▶）。无需安装任何库。
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """模拟数据集"240人班级" —— 第03-08单元统一使用 seed=2026，
    与仓库中的 lop_hoc_240.csv/.sav 完全一致（非真实数据，但100%可复现）。"""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["男", "女"], n)
    method = rng.choice(["传统教学", "项目式教学"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "项目式教学")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "项目式教学") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("已生成 df：", df.shape)

## 1. 清理 `study_hours` 的离群值（与第04/05单元相同）

In [ ]:
mask = df["study_hours"] > 20
df.loc[mask, "study_hours"] = df.loc[mask, "study_hours"] / 10

## 2. Pearson r：每周学习时长 vs 数学成绩

In [ ]:
r, p = stats.pearsonr(df["study_hours"], df["math"])
print(f"Pearson r={r:.3f}, r²={r**2:.3f}, p={p:.2e}")

#### 📤 实际输出
r≈0.582, r²≈0.338（能解释约34%的数学成绩变异），p<0.001。✅ 与课程一致。

### ❓ 常见问题：「学习时长增加1小时，数学成绩会提高多少——反过来，数学成绩提高1分，学习时长又会增加多少？」
**不是这样计算的。** r 只能告诉你线性关系的强度，不能回答"X每变化1个单位，Y会变化多少"——那是回归斜率（b）要回答的问题，是一个完全不同的量，而且**不对称**。公式：b(Y对X) = r × (Y的标准差 / X的标准差)。由于 SD(Y)/SD(X) 与 SD(X)/SD(Y) 通常不相等（除非两个变量标准差恰好相同），两个方向的回归会得出不同的数字——下面用真实数字演示。

In [ ]:
sx = df["study_hours"].std(ddof=1); sy = df["math"].std(ddof=1)
b_math_ex_hours = r * sy / sx   # 学习时长+1小时 -> 数学成绩变化多少
b_hours_ex_math = r * sx / sy   # 数学成绩+1分 -> 学习时长变化多少
print(f"学习时长标准差={sx:.3f}  数学成绩标准差={sy:.3f}")
print(f"每周学习时长+1小时 -> 数学成绩平均变化 {b_math_ex_hours:.2f} 分")
print(f"数学成绩+1分 -> 每周学习时长平均变化 {b_hours_ex_math:.3f} 小时（约{b_hours_ex_math*60:.0f}分钟）")

#### 📤 实际输出 —— 这正是「两个方向如何」这个问题的直接答案
每周学习时长+1小时 → 数学成绩平均提高约**2.51分**。但数学成绩+1分 → 学习时长平均只增加约**0.135小时（约8分钟）**。这两个数字并不是彼此的倒数，差距很大（2.51 vs 0.135）——因为两个变量的单位完全不同（小时 vs 分数），标准差（分散程度）也不同。r=0.582 是唯一一个不随方向改变的数字；而"X增加多少，Y相应增加多少"这个问题，永远要先说清楚把哪个变量当作因变量。

## 3. Spearman rho —— 与Pearson对比

In [ ]:
rho, p_rho = stats.spearmanr(df["study_hours"], df["math"])
print(f"Spearman rho={rho:.3f}, p={p_rho:.2e}")

#### 📤 实际输出
rho≈0.554 —— 与Pearson r较为接近，因为两者关系相当线性。**这个方法和Pearson有什么不同？**Spearman rho回答的是同一个问题（两个变量是否倾向于同增同减），但依据的是排名而非原始数值——因此不要求关系必须是完美直线，只需「单调同向」（一个变量增大时另一个也有增大的趋势，不必按固定比例）。适用于怀疑存在强离群值或曲线但单调的关系时。

In [ ]:
fig, ax = plt.subplots()
sns.regplot(data=df, x="study_hours", y="math", ax=ax, scatter_kws={"alpha":.5})
ax.set_title("每周学习时长 vs 数学成绩（n=240，已处理1个离群值）")
plt.show()

## 4. 偏相关 —— 真实PISA数据（195所学校）
**这个方法用来回答什么问题？**偏相关回答的是："如果'固定住'第三个变量Z的影响，X和Y之间真实剩下的关系还有多少？"——当怀疑观察到的X-Y相关，部分（甚至全部）其实是因为两者都与Z有关，而非X和Y直接相关时使用。

直接从GitHub获取学校层级数据集 —— 变量 `EDUSHORT`（教学材料短缺）、`sci_mean`（学校科学平均分）、`STAFFSHORT`（人员短缺，疑似混淆变量）。

In [ ]:
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
d = truong[["EDUSHORT", "sci_mean", "STAFFSHORT"]].dropna()
print("n =", len(d))

r_xy = stats.pearsonr(d["EDUSHORT"], d["sci_mean"])[0]
r_xz = stats.pearsonr(d["EDUSHORT"], d["STAFFSHORT"])[0]
r_yz = stats.pearsonr(d["sci_mean"], d["STAFFSHORT"])[0]
r_partial = (r_xy - r_xz*r_yz) / np.sqrt((1-r_xz**2)*(1-r_yz**2))
print(f"r(EDUSHORT, 科学分数) 原始 = {r_xy:.3f}")
print(f"r(EDUSHORT, STAFFSHORT)   = {r_xz:.3f}")
print(f"r(科学分数, STAFFSHORT)   = {r_yz:.3f}")
print(f"偏相关系数                = {r_partial:.3f}")

#### 📤 实际输出
EDUSHORT × 科学分数的原始相关：r=-0.039（非常弱）。r(EDUSHORT, STAFFSHORT)≈0.61（较高 —— 疑似混淆变量）。r(科学分数, STAFFSHORT)≈0.002（接近0）。控制STAFFSHORT后的偏相关：r=-0.051 —— **几乎没有变化**。

**为何几乎没有变化？**只有当控制变量Z与两个主变量都真正相关时，偏相关公式才会明显改变结果。此处STAFFSHORT与科学分数几乎无关（r≈0.002）—— 因此"剔除"它并不会明显改变原始结果。

## 5. 完整演示：回归的不对称性 —— EDULEAD ↔ NEGSCLIM
这是对以下问题的直接回答：*"如果教育领导力（Educational Leadership）提高1分使消极校园氛围（Negative School Climate）降低0.06分，那么消极校园氛围提高1分，是否会使教育领导力降低一个巨大的数字（比如'十几分'）？"*

**简短回答：不会 —— 但两个方向确实不同，只是按一个有明确公式的比例不同，而不是随意的、"反向放大"式的不同。**

In [ ]:
d2 = truong[["EDULEAD", "NEGSCLIM"]].dropna()
r2, p2 = stats.pearsonr(d2["EDULEAD"], d2["NEGSCLIM"])
sx2 = d2["EDULEAD"].std(ddof=1); sy2 = d2["NEGSCLIM"].std(ddof=1)
b_neg_ex_edu = r2 * sy2 / sx2   # EDULEAD+1 -> NEGSCLIM变化多少
b_edu_ex_neg = r2 * sx2 / sy2   # NEGSCLIM+1 -> EDULEAD变化多少
print(f"n={len(d2)}, r={r2:.4f}, p={p2:.4f}（p>0.05 -> 不具统计显著性，见下方提示）")
print(f"EDULEAD标准差={sx2:.4f}  NEGSCLIM标准差={sy2:.4f}")
print(f"EDULEAD +1分  -> NEGSCLIM 平均变化 {b_neg_ex_edu:+.4f} 分")
print(f"NEGSCLIM +1分 -> EDULEAD 平均变化  {b_edu_ex_neg:+.4f} 分")
print(f"两个斜率之比 = {b_neg_ex_edu/b_edu_ex_neg:.3f}  =  (NEGSCLIM标准差 / EDULEAD标准差)² = {(sy2/sx2)**2:.3f}")

#### 📤 实际输出 —— 以及正确的解读方式
r=-0.057，**p=0.427 → 不具统计显著性**（这个关系弱到可能只是抽样误差——在此展示纯粹是为了说明不对称性背后的数学原理，**不是**为了断言"领导力越强越能降低消极氛围"——以这个p值来看，证据并不充分）。

就"两个方向如何比较"这个算术问题而言：EDULEAD +1分 → NEGSCLIM 平均变化**-0.085分**。NEGSCLIM +1分 → EDULEAD 平均变化**-0.038分**。这两个数字恰好相差**2.21倍**——而2.21正好等于（NEGSCLIM标准差 / EDULEAD标准差)²。**不是直觉有时猜测的"十几倍"**——两个回归方向之间的差距，始终受两个变量标准差之比的平方所限制，不是一个随意的数字。

**需要记住的通用规则：**
1. 相关系数r：**只有一个数字**，对称，不随方向改变。
2. 回归斜率b：**有两个不同的数字**，取决于把哪个变量当作因变量——公式：b(Y对X) = r×Y的标准差/X的标准差。
3. 两个斜率之积恒等于r²：b(Y对X) × b(X对Y) = r² —— 可用来快速检验两个斜率算得对不对。
4. 只有当 X的标准差 = Y的标准差 时，两个斜率才会相等——两个用不同量表测量的变量很少出现这种情况。

In [ ]:
# 验证"斜率之积=r平方"这一恒等式
print(f"b_yx * b_xy = {b_neg_ex_edu * b_edu_ex_neg:.5f}")
print(f"r^2         = {r2**2:.5f}")
print("两者相等 -> 确认公式正确，不是巧合。")